# C3 - Day 2: Build the Resumption Dataset
**AnchorAI | Component 3: Cognitive Checkpointing**

Today we turn the raw responses into a clean table where **one row = one interrupted trial**, with:
* a **target**: how much the interruption slowed the participant down (resumption cost),
* **features**: only information that exists *before* the person resumes,
* a **participant-wise split** so that no person appears in both training and test data.

> Run on **CPU runtime** in Google Colab.

## Step 0 - Mount Drive and set paths

In [ ]:
import os, csv, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/C3'
    print('Running in Colab - Drive mounted.')
except ImportError:
    BASE = os.environ.get('C3_BASE', r'C:\Users\user\OneDrive\Documents\AnchorAI\AnchorAI\components\c3_checkpointing')
    print('Running locally.')

DATA_DIR   = os.path.join(BASE, 'data', 'raw', 'hirsch')
PROC_DIR   = os.path.join(BASE, 'data', 'processed')
OUTPUT_DIR = os.path.join(BASE, 'outputs')
os.makedirs(PROC_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Raw data folder:', DATA_DIR)
print('Files:', sorted(os.listdir(DATA_DIR)))

## Step 1 - Load the three experiments
Same loader as Day 1: parse the quoted rows, make column names identical, keep 9 columns.

In [ ]:
def load_hirsch_csv(path):
    """Read a Hirsch CSV whose every row is wrapped in outer double-quotes."""
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.read().splitlines()

    def parse(line):
        line = line.strip()
        if line.startswith('"') and line.endswith('"'):
            line = line[1:-1].replace('""', '"')
        return next(csv.reader([line]))

    header = parse(lines[0])
    rows = [parse(l) for l in lines[1:] if l.strip()]
    return pd.DataFrame(rows, columns=header)


KEEP = ['pp', 'block', 'blocktype', 'trial', 'trialtype', 'task', 'sequence', 'RT', 'error']
frames = []
for exp, fname in [('Exp1', 'raw-data_Exp1.csv'), ('Exp2', 'raw-data_Exp2.csv'), ('Exp3', 'raw-data_Exp3.csv')]:
    d = load_hirsch_csv(os.path.join(DATA_DIR, fname))
    d = d.rename(columns={'bBlock': 'block', 'Trialtype': 'trialtype'})   # make the names identical
    d = d[KEEP].copy()
    d['experiment'] = exp
    d['row_order'] = np.arange(len(d))
    frames.append(d)

raw = pd.concat(frames, ignore_index=True)
for c in KEEP:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')
raw['pp_id'] = raw['experiment'] + '_' + raw['pp'].astype(int).astype(str)   # participant id, unique across experiments

print(raw.groupby('experiment').agg(rows=('RT', 'size'), participants=('pp_id', 'nunique')))
print('Total rows:', len(raw), '| total participants:', raw['pp_id'].nunique())

## Step 2 - One row per trial (experimental blocks only)
Practice blocks are dropped. Then each trial's 3-4 response rows are reshaped into one row: `rt0` = interruption task, `rt1`, `rt2` = subtasks before the interruption, `rt3` = **the resumption step**. A trial counts as *interrupted* if it contains a task-0 response.

In [ ]:
KEY = ['experiment', 'pp_id', 'block', 'trial']
exp_rows = raw[raw['blocktype'] == 2]                       # experimental blocks only

rt_wide  = exp_rows.pivot(index=KEY, columns='task', values='RT').add_prefix('rt')       # rt0, rt1, rt2, rt3 (rt4)
err_wide = exp_rows.pivot(index=KEY, columns='task', values='error').add_prefix('err')   # err0, err1, ...
info     = exp_rows.groupby(KEY).agg(sequence=('sequence', 'first'), trialtype=('trialtype', 'first'))

trials = pd.concat([rt_wide, err_wide, info], axis=1).reset_index()
trials['interrupted'] = trials['rt0'].notna()                # True if the interruption task (task 0) really happened
print('Trials (experimental blocks):', len(trials))
print(trials['interrupted'].value_counts().rename({True: 'interrupted', False: 'uninterrupted'}))

## Step 3 - Valid reaction-time window (outlier trimming)
Some reaction times are impossible or extreme (e.g. several minutes: the participant looked away).
* Lower limit: **200 ms**. Faster than this is an accidental key press, not a real decision.
* Upper limit: **Q3 + 3 x IQR**, computed **separately for each experiment** because the experiments have different task lengths. We use 3x (not 1.5x) because reaction time is naturally right-skewed and we do not want to throw away genuinely slow but valid responses.

IQR = Q3 - Q1 (the spread of the middle 50% of values).

In [ ]:
limits = {}
for e, g in exp_rows.groupby('experiment'):
    q1, q3 = g['RT'].quantile(0.25), g['RT'].quantile(0.75)
    limits[e] = (200.0, float(q3 + 3 * (q3 - q1)))
    print(f'{e}: Q1={q1:.0f}, Q3={q3:.0f}, valid window = {limits[e][0]:.0f} - {limits[e][1]:.0f} ms')

def in_window(df, col):
    lo = df['experiment'].map(lambda e: limits[e][0])
    hi = df['experiment'].map(lambda e: limits[e][1])
    return (df[col] >= lo) & (df[col] <= hi)

## Step 4 - The target: resumption cost
**Resumption cost** = how much slower a person is on subtask 3 right after an interruption, compared with how fast *that same person* normally does subtask 3 without an interruption.

* `baseline_rt` = the participant's **median** RT on subtask 3 in their *uninterrupted* trials (correct responses only), computed separately for `sequence` = 1 and 2 because repeat/switch trials have different speeds.
* `cost = resumption_rt - baseline_rt` (in ms).

**Why a baseline?** Some people are simply faster than others. Subtracting the person's own normal speed leaves the effect of the **interruption**, which is what C3 is about. A positive cost means the interruption slowed the person down.

In [ ]:
unint = trials[(~trials['interrupted']) & (trials['err3'] == 0) & in_window(trials, 'rt3')]
baseline_seq = unint.groupby(['pp_id', 'sequence'])['rt3'].median().rename('baseline_rt').reset_index()
baseline_pp  = unint.groupby('pp_id')['rt3'].median().rename('baseline_pp').reset_index()

calm = trials[(~trials['interrupted']) & (trials['err1'] == 0) & (trials['err2'] == 0)
              & in_window(trials, 'rt1') & in_window(trials, 'rt2')]
speed = ((calm['rt1'] + calm['rt2']) / 2).groupby(calm['pp_id']).median().rename('pp_speed').reset_index()

# funnel: how many interrupted trials survive each cleaning step
S = trials[trials['interrupted']].copy()
funnel = [('Interrupted trials (experimental blocks)', len(S))]
S = S[S['err3'] == 0];                                   funnel.append(('Resumption step answered correctly', len(S)))
for c in ['rt0', 'rt1', 'rt2', 'rt3']:
    S = S[in_window(S, c)]
funnel.append(('All four reaction times inside valid window', len(S)))
S = S.merge(baseline_seq, on=['pp_id', 'sequence'], how='left').merge(baseline_pp, on='pp_id', how='left')
S['baseline_rt'] = S['baseline_rt'].fillna(S['baseline_pp'])
S = S.merge(speed, on='pp_id', how='left').dropna(subset=['baseline_rt', 'pp_speed']).reset_index(drop=True)
funnel.append(('Participant has a baseline', len(S)))

for name, n in funnel:
    print(f'{n:>7,}  {name}')

S['resumption_rt'] = S['rt3']
S['cost'] = S['resumption_rt'] - S['baseline_rt']
print('\nResumption cost (ms):')
print(S['cost'].describe().round(1))
print(f'\nShare of trials with positive cost: {(S["cost"] > 0).mean() * 100:.1f}%')

## Step 5 - Features: only what is known *before* the person resumes
A feature must be available at the moment the interruption ends. We never use anything from the resumption step itself (its reaction time, its error).

| Group | Feature | Meaning | C3 link |
|-------|---------|---------|---------|
| design | `sequence` | last subtask repeats (1) or switches (2) | structure of the task |
| design | `block`, `trial` | position in the session | fatigue / practice |
| design | `experiment` | which experiment (Exp3 has 4 subtasks) | task complexity |
| interruption | `interruption_rt` | time spent on the interruption task | interruption length |
| interruption | `interruption_error` | error on the interruption task | how demanding it was |
| pace | `rt_subtask1`, `rt_subtask2` | speed before the interruption | work pace |
| pace | `pre_errors` | errors before the interruption | state of the work |
| person | `pp_speed` | the person's usual speed (uninterrupted trials) | individual differences |

In [ ]:
S = S.rename(columns={'rt0': 'interruption_rt', 'err0': 'interruption_error',
                      'rt1': 'rt_subtask1', 'rt2': 'rt_subtask2'})
S['pre_errors'] = S['err1'] + S['err2']

FEATURES = ['sequence', 'block', 'trial', 'experiment',
            'interruption_rt', 'interruption_error',
            'rt_subtask1', 'rt_subtask2', 'pre_errors',
            'pp_speed']
S[FEATURES + ['resumption_rt', 'baseline_rt', 'cost']].head()

## Step 6 - Participant-wise split
Each participant contributes many trials. If we split rows randomly, the same person ends up in both training and test data, the model partly recognises the person, and accuracy looks better than it really is (**data leakage**).

So we split **participants**:
* 80% of participants -> `train`, 20% -> `test` (never touched until the final evaluation).
* Training participants are further divided into **5 folds** (`fold` 0-4) for cross-validation: all trials of one person always stay in the same fold. Day 3 and Day 4 use these same folds so all models are compared fairly.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(S, groups=S['pp_id']))
S['split'] = 'train'
S.loc[test_idx, 'split'] = 'test'

rng = np.random.RandomState(42)
train_pp = np.array(sorted(S.loc[S['split'] == 'train', 'pp_id'].unique()))
rng.shuffle(train_pp)
fold_of = {p: i % 5 for i, p in enumerate(train_pp)}
S['fold'] = S['pp_id'].map(fold_of).fillna(-1).astype(int)       # -1 = test participants

tr_pp, te_pp = set(S.loc[S['split'] == 'train', 'pp_id']), set(S.loc[S['split'] == 'test', 'pp_id'])
print(f'Train: {(S["split"] == "train").sum():,} trials | {len(tr_pp)} participants')
print(f'Test : {(S["split"] == "test").sum():,} trials | {len(te_pp)} participants')
print('Participants in both train and test (must be 0):', len(tr_pp & te_pp))
print('\nTrials per fold:', S[S['fold'] >= 0].groupby('fold').size().to_dict())

## Step 7 - Low / Medium / High labels
We turn the cost into three classes using **terciles** (the 33rd and 67th percentiles):
* **Low** = smallest third of costs (easy resumption)
* **Medium** = middle third
* **High** = largest third (hard resumption)

The cut points are computed on the **training participants only**, then applied to everybody. If we computed them on all data, the test set would slightly influence how the labels are defined.

This is a *preliminary proxy label*: it describes how hard resumption was relative to the other trials in this lab dataset, not an absolute clinical threshold.

In [ ]:
q33, q67 = S.loc[S['split'] == 'train', 'cost'].quantile([1 / 3, 2 / 3])
print(f'Cut points from training data: Low <= {q33:.0f} ms < Medium <= {q67:.0f} ms < High')

LABEL_NAMES = ['Low', 'Medium', 'High']
S['label'] = np.where(S['cost'] <= q33, 0, np.where(S['cost'] <= q67, 1, 2))
S['label_name'] = S['label'].map(dict(enumerate(LABEL_NAMES)))

print('\nLabel counts by split:')
print(pd.crosstab(S['split'], S['label_name'])[LABEL_NAMES])
print('\nLabel counts by experiment:')
print(pd.crosstab(S['experiment'], S['label_name'])[LABEL_NAMES])

plt.figure(figsize=(8, 4))
plt.hist(S['cost'], bins=70, color='steelblue', edgecolor='white')
plt.axvline(q33, color='orange', ls='--', label=f'33rd percentile ({q33:.0f} ms)')
plt.axvline(q67, color='red', ls='--', label=f'67th percentile ({q67:.0f} ms)')
plt.axvline(0, color='black', lw=1)
plt.title('Resumption cost (interrupted RT minus own baseline)')
plt.xlabel('Cost (ms)'); plt.ylabel('Trials'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'resumption_cost_distribution.png'), dpi=150)
plt.show()

## Step 8 - Save for Day 3 and Day 4
One CSV (one row per interrupted trial) plus a small `meta.json` that records every decision.

In [ ]:
cols = ['pp_id', 'experiment', 'split', 'fold'] + FEATURES + ['resumption_rt', 'baseline_rt', 'cost', 'label', 'label_name']
S[cols].to_csv(os.path.join(PROC_DIR, 'samples.csv'), index=False)

meta = {
    'features': FEATURES,
    'label_names': LABEL_NAMES,
    'tercile_cut_points_ms': {'low_upper': float(q33), 'medium_upper': float(q67)},
    'rt_valid_window_ms': {e: [lo, hi] for e, (lo, hi) in limits.items()},
    'funnel': [{'step': s, 'trials': int(n)} for s, n in funnel],
    'n_train_trials': int((S['split'] == 'train').sum()),
    'n_test_trials': int((S['split'] == 'test').sum()),
    'n_train_participants': len(tr_pp),
    'n_test_participants': len(te_pp),
    'random_state': 42,
}
with open(os.path.join(PROC_DIR, 'meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved to', PROC_DIR, ':', sorted(os.listdir(PROC_DIR)))
print(json.dumps(meta, indent=2))

## Summary - Day 2 (numbers are printed above)
| Step | What we did | Why |
|------|-------------|-----|
| Experimental blocks only | dropped practice | practice RTs are not representative |
| One row per trial | reshaped responses into trials | the interruption happens inside a trial |
| Valid RT window | 200 ms to Q3 + 3 x IQR per experiment | remove impossible / extreme values |
| Target | resumption cost = interrupted RT - own baseline | isolates the effect of the interruption |
| Features | only information known before resuming | no leakage from the resumption step |
| Split | participants into train / test and 5 folds | no person in two sets |
| Labels | terciles of cost, cut points from training data | Low / Medium / High difficulty |

**Next: Day 3 - Logistic Regression baseline.**